# Preparación de Datos Desde Cero
## De los libros anuales de SIMA a los datos de análisis, paso a paso

Este notebook documenta el proceso completo de construcción de los datos para
garantizar replicabilidad: cómo se leen los libros ORIGINALES de SIMA, cómo se
consolidan por zona de monitoreo, cuántos datos faltan, cómo se imputan los huecos
(dos vías independientes) y cómo se construye el calendario escolar OFICIAL (fijo,
por fecha, tomado de los acuerdos de la SEP publicados en el Diario Oficial de la
Federación; no derivado del dataset). Sirve también como receta para incorporar un
libro nuevo de SIMA en el futuro.

**Data access / Acceso a los datos.** The hourly measurements belong to SIMA (Sistema
Integral de Monitoreo Ambiental, Gobierno del Estado de Nuevo León). SIMA provided the
annual workbooks to the authors for this study on the condition that they are used in
the analysis but not redistributed. This repository therefore contains no measurement
series at any resolution: neither the workbooks in `data_raw/` nor the consolidated
hourly, station, zone, day-window or gap files that this notebook and notebook 01 build
from them (all of them are git-ignored). What it does contain is the complete code, the
official SEP school calendar, the trained MIMA model and every derived result reported
in the article (estimates, diagnostics, tables and figures). To rebuild everything,
request the hourly data from SIMA (https://aire.nl.gob.mx/), place the three annual
workbooks in `data_raw/` as `BD 2023.xlsx`, `BD 2024.xlsx` and `BD 2025.xlsx` (one sheet
per station named with its SIMA code, first column `Fecha y hora` or `date`, one column
per parameter) and run this notebook; it stops with a clear message if they are absent.
The MIMA series is produced afterwards by `data_processed/mima/mima_pipeline.py`.

Las mediciones horarias pertenecen a SIMA (Sistema Integral de Monitoreo Ambiental,
Gobierno del Estado de Nuevo León). SIMA entregó los libros anuales a los autores para
este estudio con la condición de usarlos en el análisis pero no difundirlos. Por eso
este repositorio no contiene series de mediciones en ninguna resolución: ni los libros
de `data_raw/` ni los archivos horarios, por estación, por zona, por ventana del día o
de brechas que este notebook y el 01 construyen a partir de ellos (todos ignorados por
git). Lo que sí contiene es el código completo, el calendario oficial de la SEP, el
modelo MIMA entrenado y todos los resultados derivados que reporta el artículo
(estimaciones, diagnósticos, tablas y figuras). Para reconstruir todo, solicite los
datos horarios a SIMA (https://aire.nl.gob.mx/), coloque los tres libros anuales en
`data_raw/` como `BD 2023.xlsx`, `BD 2024.xlsx` y `BD 2025.xlsx` (una hoja por estación
con su código de SIMA, primera columna `Fecha y hora` o `date`, una columna por
parámetro) y corra este notebook; se detiene con un mensaje claro si no están. La serie
de MIMA se produce después con `data_processed/mima/mima_pipeline.py`.

**Role in the analysis / Papel en el análisis.** Data preparation: the pipeline from the SIMA workbooks to the consolidated hourly bases, the imputations and the official calendar. / Preparación de datos: la cadena desde los libros de SIMA hasta las bases horarias consolidadas, las imputaciones y el calendario oficial. The analytical hierarchy of the whole repository is stated in the README. / La jerarquía analítica de todo el repositorio está en el README.

## 1. Configuración inicial

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Configuración de estilo
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

contaminantes = ['CO', 'NO2', 'O3', 'PM10', 'PM2.5']

raw_dir = Path('../data_raw')
output_dir = Path('../data_processed')
censo_dir = output_dir / 'censo_datos'
censo_dir.mkdir(parents=True, exist_ok=True)

print("Librerías importadas exitosamente")

Librerías importadas exitosamente


## 2. Lectura de los libros ORIGINALES de SIMA

Tres libros anuales (`BD 2023.xlsx`, `BD 2024.xlsx`, `BD 2025.xlsx`), todos con el
mismo formato: una hoja por estación física (la clave de sector de SIMA es el nombre
de la hoja), la primera columna es la fecha-hora y hay una columna por parámetro
(CO, NO, NO2, NOX, O3, PM10, PM2.5, PRS, RAINF, RH, SO2, SR, TOUT, WSR, WDR). El
análisis usa los cinco contaminantes criterio con cobertura completa en la red:
CO, NO2, O3, PM10 y PM2.5.

Todo se lleva a formato largo: (archivo, estacion_fisica, fecha-hora, parametro, valor).

In [2]:
# Claves de hoja -> nombre completo de la estación física (mismos códigos de SIMA)
codigos_estacion = {'SE': 'SURESTE', 'CE': 'CENTRO', 'SO': 'SUROESTE', 'NE': 'NORESTE',
                    'NO': 'NOROESTE', 'NTE': 'NORTE', 'NE2': 'NORESTE2', 'SE2': 'SURESTE2',
                    'SE3': 'SURESTE 3', 'NO2': 'NOROESTE 2', 'NO3': 'NOROESTE 3',
                    'NTE2': 'NORTE 2', 'SO2': 'SUROESTE2', 'SUR': 'SUR', 'NE3': 'NORESTE 3'}

# Mapeo estación física -> zona de monitoreo (las familias de sector de SIMA)
mapeo_zona = {'SUR': 'SUR', 'SURESTE': 'SUR', 'SURESTE2': 'SUR', 'SURESTE 3': 'SUR',
              'SUROESTE': 'SUROESTE2', 'SUROESTE2': 'SUROESTE2',
              'NORTE': 'NORTE2', 'NORTE 2': 'NORTE2',
              'CENTRO': 'CENTRO',
              'NORESTE': 'NORESTE3', 'NORESTE2': 'NORESTE3', 'NORESTE 3': 'NORESTE3',
              'NOROESTE': 'NOROESTE3', 'NOROESTE 2': 'NOROESTE3', 'NOROESTE 3': 'NOROESTE3'}


def leer_libro_sima(ruta, nombre_archivo):
    """
    Lee un libro anual de SIMA (una hoja por estación física) y lo devuelve en
    formato largo. Los encabezados se normalizan por si traen unidades o el nombre
    de la columna de fecha cambia entre libros ('Fecha y hora' o 'date').
    """
    hojas = pd.read_excel(ruta, sheet_name=None, engine='calamine')
    piezas = []
    for hoja, df in hojas.items():
        df = df.copy()
        df.columns = [str(c).split(' (')[0].strip() for c in df.columns]
        df = df.rename(columns={'Fecha y hora': 'date', 'fecha': 'date', 'Fecha': 'date'})
        if 'date' not in df.columns:
            continue
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        df = df.dropna(subset=['date'])
        presentes = [c for c in contaminantes if c in df.columns]
        largo = df.melt(id_vars=['date'], value_vars=presentes,
                        var_name='parametro', value_name='valor')
        largo['valor'] = pd.to_numeric(largo['valor'], errors='coerce')
        clave = hoja.strip().upper()
        largo['estacion_fisica'] = codigos_estacion.get(clave, clave)
        piezas.append(largo)
    todo = pd.concat(piezas, ignore_index=True)
    todo['archivo'] = nombre_archivo
    return todo


archivos_crudos = {'2023': raw_dir / 'BD 2023.xlsx',
                   '2024': raw_dir / 'BD 2024.xlsx',
                   '2025': raw_dir / 'BD 2025.xlsx'}
hay_crudos = all(ruta.exists() for ruta in archivos_crudos.values())

print("=" * 70)
print("LECTURA DE ARCHIVOS ORIGINALES")
print("=" * 70)
if hay_crudos:
    piezas_crudas = []
    for nombre, ruta in archivos_crudos.items():
        df = leer_libro_sima(ruta, nombre)
        piezas_crudas.append(df)
        print(f"  {nombre}: {len(df):,} filas | {df['estacion_fisica'].nunique()} estaciones físicas | "
              f"{df['date'].min().date()} a {df['date'].max().date()}")
    crudos = pd.concat(piezas_crudas, ignore_index=True)
    crudos['zona'] = crudos['estacion_fisica'].map(mapeo_zona)
    sin_zona = crudos[crudos['zona'].isna()]['estacion_fisica'].unique()
    print(f"  TOTAL: {len(crudos):,} filas (con valores nulos incluidos)")
    print(f"  Estaciones físicas sin zona asignada: {list(sin_zona) if len(sin_zona) else 'ninguna'}")
    # Base por estación física: una fila por (estación, fecha-hora, parámetro). Si un
    # sello de tiempo viniera repetido dentro de un libro, se promedia.
    base_estaciones = (crudos.dropna(subset=['valor', 'zona'])
                       .groupby(['estacion_fisica', 'zona', 'date', 'parametro'], as_index=False)['valor'].mean())
    base_estaciones.to_csv(output_dir / 'base_horaria_estaciones_fisicas.csv.gz',
                           index=False, compression='gzip')
    print(f"\nGuardado en: {output_dir / 'base_horaria_estaciones_fisicas.csv.gz'}")
else:
    raise FileNotFoundError(
        "Los libros anuales de SIMA no están en data_raw/. No se redistribuyen: solicítelos a SIMA "
        "(https://aire.nl.gob.mx/) y colóquelos como 'BD 2023.xlsx', 'BD 2024.xlsx' y 'BD 2025.xlsx'. / "
        "The SIMA annual workbooks are not in data_raw/. They are not redistributed: request them from "
        "SIMA (https://aire.nl.gob.mx/) and place them as 'BD 2023.xlsx', 'BD 2024.xlsx' and 'BD 2025.xlsx'.")

print(f"\nBase por estación física: {len(base_estaciones):,} filas | "
      f"{base_estaciones['estacion_fisica'].nunique()} estaciones | "
      f"{base_estaciones['date'].min()} a {base_estaciones['date'].max()}")

LECTURA DE ARCHIVOS ORIGINALES


  2023: 656,850 filas | 15 estaciones físicas | 2023-01-01 a 2023-12-31


  2024: 658,705 filas | 15 estaciones físicas | 2024-01-01 a 2024-12-31


  2025: 656,890 filas | 15 estaciones físicas | 2025-01-01 a 2025-12-31


  TOTAL: 1,972,445 filas (con valores nulos incluidos)
  Estaciones físicas sin zona asignada: ninguna



Guardado en: ../data_processed/base_horaria_estaciones_fisicas.csv.gz

Base por estación física: 1,751,750 filas | 15 estaciones | 2023-01-01 00:00:00 a 2025-12-31 23:00:00


## 3. La red se organiza por SECTORES

SIMA nombra sus 15 estaciones físicas por familias de sector (SURESTE, SURESTE2,
SURESTE 3, etc.): estaciones de la misma familia monitorean la misma zona de la mancha
urbana, y según la propia caracterización de la red cada estación representa un radio
de unos 2 a 3 km. El análisis define 6 zonas de monitoreo agrupando cada familia;
abajo se verifica empíricamente que las estaciones de una misma zona se mueven juntas
(esa coherencia es la que justifica promediarlas) y que agrupar además estabiliza la
cobertura cuando una estación individual falla.

In [3]:
tabla_zonas = (base_estaciones.groupby(['zona', 'estacion_fisica'])
               .agg(filas=('valor', 'count'),
                    desde=('date', lambda s: s.min().date()),
                    hasta=('date', lambda s: s.max().date())).reset_index())
print("=" * 70)
print("COMPOSICIÓN DE CADA ZONA (estaciones físicas que agrupa)")
print("=" * 70)
print(tabla_zonas.to_string(index=False))
tabla_zonas.to_csv(censo_dir / 'zonas_y_estaciones_fisicas.csv', index=False)
print(f"\nGuardado en: {censo_dir / 'zonas_y_estaciones_fisicas.csv'}")

COMPOSICIÓN DE CADA ZONA (estaciones físicas que agrupa)
     zona estacion_fisica  filas      desde      hasta
   CENTRO          CENTRO 123007 2023-01-01 2025-12-31
 NORESTE3         NORESTE 123828 2023-01-01 2025-12-31
 NORESTE3       NORESTE 3  99063 2023-01-01 2025-12-31
 NORESTE3        NORESTE2 107096 2023-01-01 2025-12-31
NOROESTE3        NOROESTE 100719 2023-01-01 2025-12-31
NOROESTE3      NOROESTE 2 121082 2023-01-01 2025-12-31
NOROESTE3      NOROESTE 3  88116 2023-01-01 2025-12-31
   NORTE2           NORTE 124771 2023-01-01 2025-12-31
   NORTE2         NORTE 2 126179 2023-01-01 2025-12-31
      SUR             SUR 122107 2023-01-01 2025-12-31
      SUR         SURESTE 120965 2023-01-01 2025-12-31
      SUR       SURESTE 3 123982 2023-01-01 2025-12-31
      SUR        SURESTE2 122467 2023-01-01 2025-12-31
SUROESTE2        SUROESTE 123431 2023-01-01 2025-12-31
SUROESTE2       SUROESTE2 124937 2023-01-01 2025-12-31

Guardado en: ../data_processed/censo_datos/zonas_y_estaciones_

## 4. Consolidación a la base horaria por zona

Para cada (zona, fecha-hora, contaminante) se promedian las estaciones físicas de la
zona. Antes de promediar se verifica que el promedio sea legítimo: las estaciones de
una misma zona deben contar la misma historia. En el notebook 02 se muestra además
que la elección media-vs-mediana es inocua (cambia 2-3% la celda diaria, un orden de
magnitud menos que los efectos que se contrastan).

In [4]:
base_horaria = (base_estaciones.groupby(['zona', 'date', 'parametro'], as_index=False)['valor'].mean()
                .rename(columns={'zona': 'estacion'}))

print("=" * 70)
print("BASE HORARIA CONSOLIDADA POR ZONA")
print("=" * 70)
print(f"Filas: {len(base_horaria):,} | zonas: {base_horaria['estacion'].nunique()} | "
      f"{base_horaria['date'].min()} a {base_horaria['date'].max()}")

# Coherencia intra-zona: correlación (en días) entre las estaciones físicas de cada zona.
# Si las estaciones de una zona no se movieran juntas, promediarlas sería mezclar peras
# con manzanas; correlaciones altas legitiman la consolidación.
diario_fisico = (base_estaciones
                 .assign(fecha=lambda d: d['date'].dt.normalize())
                 .groupby(['zona', 'estacion_fisica', 'fecha', 'parametro'])['valor'].mean()
                 .reset_index())
print("\nCoherencia intra-zona (correlación diaria media entre estaciones de la misma zona):")
coherencia = []
for zona_v, sub_z in diario_fisico.groupby('zona'):
    if sub_z['estacion_fisica'].nunique() < 2:
        print(f"  {zona_v}: 1 estación (sin par que comparar)")
        continue
    correlaciones = []
    for c in contaminantes:
        ancho = (sub_z[sub_z['parametro'] == c]
                 .pivot_table(index='fecha', columns='estacion_fisica', values='valor'))
        if ancho.shape[1] >= 2:
            m = ancho.corr().values
            correlaciones.append(m[np.triu_indices_from(m, k=1)].mean())
    print(f"  {zona_v}: correlación media entre pares = {np.nanmean(correlaciones):.2f}")
    coherencia.append({'zona': zona_v, 'n_estaciones': sub_z['estacion_fisica'].nunique(),
                       'correlacion_media_pares': round(float(np.nanmean(correlaciones)), 3)})
pd.DataFrame(coherencia).to_csv(censo_dir / 'coherencia_intrazona.csv', index=False)

base_horaria.to_csv(output_dir / 'base_horaria_zonas.csv.gz', index=False, compression='gzip')
print(f"\nGuardado en: {output_dir / 'base_horaria_zonas.csv.gz'}")

BASE HORARIA CONSOLIDADA POR ZONA
Filas: 772,426 | zonas: 6 | 2023-01-01 00:00:00 a 2025-12-31 23:00:00



Coherencia intra-zona (correlación diaria media entre estaciones de la misma zona):
  CENTRO: 1 estación (sin par que comparar)
  NORESTE3: correlación media entre pares = 0.57
  NOROESTE3: correlación media entre pares = 0.58
  NORTE2: correlación media entre pares = 0.68
  SUR: correlación media entre pares = 0.66
  SUROESTE2: correlación media entre pares = 0.66



Guardado en: ../data_processed/base_horaria_zonas.csv.gz


## 5. La tabla de datos perdidos (¿cuánto falta y cuánto se imputa?)

Se cuenta dos veces: por zona (lo que la imputación rellena, porque las series de
análisis son las de zona) y por monitor físico, antes de consolidar. La diferencia
entre las dos cuentas es una de las razones para agrupar por zona: una hora de zona
solo falta cuando faltan todos sus monitores.

In [5]:
fecha_min, fecha_max = base_horaria['date'].min(), base_horaria['date'].max()
horas_esperadas = len(pd.date_range(fecha_min, fecha_max, freq='h'))

presentes = (base_horaria.groupby(['estacion', 'parametro'])['valor'].count()
             .unstack('parametro').reindex(columns=contaminantes))
tabla_faltantes = pd.DataFrame({
    'horas_esperadas': horas_esperadas,
    **{f'faltante_{c}_pct': ((1 - presentes[c] / horas_esperadas) * 100).round(1) for c in contaminantes}})

print("=" * 70)
print(f"DATOS PERDIDOS POR ZONA Y CONTAMINANTE ({fecha_min.date()} a {fecha_max.date()})")
print("=" * 70)
print(tabla_faltantes.to_string())
promedio_faltante = np.mean([(1 - presentes[c] / horas_esperadas).mean() for c in contaminantes]) * 100
print(f"\nPromedio global de horas faltantes (lo que la imputación rellena): {promedio_faltante:.1f}%")
tabla_faltantes.to_csv(censo_dir / 'tabla_datos_perdidos.csv')
print(f"Guardado en: {censo_dir / 'tabla_datos_perdidos.csv'}")

# La misma cuenta a nivel de monitor físico (antes de consolidar por zona). Consolidar
# reduce mucho los huecos: una hora de zona solo falta cuando faltan TODOS sus monitores.
presentes_fisicas = (base_estaciones.groupby(['estacion_fisica', 'parametro'])['valor'].count()
                     .unstack('parametro').reindex(columns=contaminantes))
faltante_fisicas = ((1 - presentes_fisicas / horas_esperadas) * 100).round(1)
faltante_fisicas.index.name = 'estacion_fisica'
print("\nDATOS PERDIDOS POR MONITOR FÍSICO Y CONTAMINANTE (antes de consolidar por zona)")
print(faltante_fisicas.to_string())
promedio_faltante_fisicas = float(faltante_fisicas.values.mean())
print(f"\nPromedio a nivel de monitor físico: {promedio_faltante_fisicas:.1f}% | "
      f"a nivel de zona (lo que la imputación rellena): {promedio_faltante:.1f}%")
faltante_fisicas.to_csv(censo_dir / 'pct_faltante_por_estacion_fisica.csv')
print(f"Guardado en: {censo_dir / 'pct_faltante_por_estacion_fisica.csv'}")

DATOS PERDIDOS POR ZONA Y CONTAMINANTE (2023-01-01 a 2025-12-31)
           horas_esperadas  faltante_CO_pct  faltante_NO2_pct  faltante_O3_pct  faltante_PM10_pct  faltante_PM2.5_pct
estacion                                                                                                             
CENTRO               26304              8.1               3.1              3.9                2.3                15.0
NORESTE3             26304              1.2               3.0              0.2                0.1                 2.9
NOROESTE3            26304              0.2               1.0              0.7                0.1                14.2
NORTE2               26304              0.2               0.2              0.7                0.3                 1.1
SUR                  26304              0.1               0.1              0.1                0.1                 0.1
SUROESTE2            26304              0.3               0.1              0.6                0.2            


DATOS PERDIDOS POR MONITOR FÍSICO Y CONTAMINANTE (antes de consolidar por zona)
parametro          CO   NO2    O3  PM10  PM2.5
estacion_fisica                               
CENTRO            8.1   3.1   3.9   2.3   15.0
NORESTE           5.6   5.2   6.3   3.5    8.7
NORESTE 3        10.7  10.0   5.7   5.4   91.6
NORESTE2         16.5  15.3   6.9   4.4   49.7
NOROESTE         14.4  12.4   8.2   4.0   78.1
NOROESTE 2        2.6  11.7   3.7   2.4   19.2
NOROESTE 3       19.1  29.5  16.7  13.1   86.5
NORTE             3.0   3.0   5.2   4.4   10.1
NORTE 2           2.5   1.1   4.5   2.4    9.8
SUR               7.5   4.7   1.6   1.9   20.1
SURESTE          13.6   1.9   2.9   2.7   19.0
SURESTE 3         9.5   2.5   2.2   2.2   12.3
SURESTE2         14.6   3.3   7.9   2.7    5.9
SUROESTE          2.8   4.1   5.0   2.7   16.3
SUROESTE2         4.2   0.7   1.6   4.6   14.0

Promedio a nivel de monitor físico: 11.2% | a nivel de zona (lo que la imputación rellena): 2.1%
Guardado en: ../data_p

## 6. Imputación de los huecos (las dos vías, implementadas desde esta base)

El análisis corre SIEMPRE con dos imputaciones independientes, para que ninguna
conclusión dependa del método de relleno:

**(a) Híbrida (estacional-local), implementada aquí mismo.** Dos pasos por serie
(zona × contaminante): los huecos cortos (hasta 6 horas) se rellenan por
interpolación temporal, que respeta la trayectoria local del día; los huecos largos
se rellenan con el perfil climatológico de la propia serie, la mediana observada
para esa combinación de mes, tipo de día (hábil o fin de semana) y hora del día.
Si un estrato no tiene datos, se cae al perfil mes × hora de la serie y, en último
caso, al de la red. Todo valor imputado se recorta al piso físico del instrumento.

**(b) MIMA:** modelo de imputación por aprendizaje profundo (SAITS, transformer de
auto-atención, con etapa CSDI) desarrollado para esta red por Gómez, Sánchez, Medina,
Montibeller y Pérez (Tecnológico de Monterrey). Se entrena sobre esta misma base
horaria consolidada con el script `../data_processed/mima/mima_pipeline.py`, que
también guarda el modelo entrenado y su auditoría con huecos artificiales (celdas
observadas que se esconden a propósito para medir qué tan bien se recuperan; el
notebook 04 la analiza). Entrenar tarda decenas de minutos en CPU, así que aquí no se
re-entrena: se carga la serie ya imputada por ese script.

In [6]:
pisos_fisicos = {'CO': 0.05, 'NO2': 0.05, 'O3': 0.5, 'PM10': 1.0, 'PM2.5': 0.5}
malla_horas = pd.date_range(fecha_min, fecha_max, freq='h')

series_imputadas = []
for (zona_v, c), sub_s in base_horaria.groupby(['estacion', 'parametro']):
    s = sub_s.set_index('date')['valor'].reindex(malla_horas)
    observada = s.notna()
    # Paso 1: huecos cortos por interpolación temporal
    s = s.interpolate(method='time', limit=6, limit_area='inside')
    # Paso 2: huecos largos por perfil climatológico mes × tipo de día × hora
    marco = pd.DataFrame({'valor': s})
    marco['mes'] = marco.index.month
    marco['habil'] = (marco.index.dayofweek < 5).astype(int)
    marco['hora'] = marco.index.hour
    perfil = (marco.dropna(subset=['valor'])
              .groupby(['mes', 'habil', 'hora'])['valor'].median())
    faltan = marco['valor'].isna()
    relleno = marco.loc[faltan, ['mes', 'habil', 'hora']].apply(
        lambda fila: perfil.get((fila['mes'], fila['habil'], fila['hora']), np.nan), axis=1)
    marco.loc[faltan, 'valor'] = relleno
    # Fallback: perfil mes × hora de la propia serie
    if marco['valor'].isna().any():
        perfil2 = marco.dropna(subset=['valor']).groupby(['mes', 'hora'])['valor'].median()
        faltan2 = marco['valor'].isna()
        marco.loc[faltan2, 'valor'] = marco.loc[faltan2, ['mes', 'hora']].apply(
            lambda fila: perfil2.get((fila['mes'], fila['hora']), np.nan), axis=1)
    marco['valor'] = marco['valor'].clip(lower=pisos_fisicos[c])
    salida_s = marco[['valor']].reset_index(names='date')
    salida_s['estacion'] = zona_v
    salida_s['parametro'] = c
    salida_s['imputada'] = (~observada.values).astype(int)
    series_imputadas.append(salida_s)

imputado_hibrido = pd.concat(series_imputadas, ignore_index=True)

# Fallback final de red (estratos vacíos en toda la serie): perfil mes × hora del contaminante
if imputado_hibrido['valor'].isna().any():
    perfil_red = (imputado_hibrido.dropna(subset=['valor'])
                  .assign(mes=lambda d: d['date'].dt.month, hora=lambda d: d['date'].dt.hour)
                  .groupby(['parametro', 'mes', 'hora'])['valor'].median())
    faltan_red = imputado_hibrido['valor'].isna()
    aux = imputado_hibrido.loc[faltan_red].assign(mes=lambda d: d['date'].dt.month,
                                                  hora=lambda d: d['date'].dt.hour)
    imputado_hibrido.loc[faltan_red, 'valor'] = aux.apply(
        lambda fila: perfil_red.get((fila['parametro'], fila['mes'], fila['hora']), np.nan), axis=1)

imputado_hibrido.to_csv(output_dir / 'datos_horarios_imputados_hibrido.csv.gz',
                        index=False, compression='gzip')
ruta_mima = output_dir / 'datos_horarios_imputados_MIMA.csv.gz'
if ruta_mima.exists():
    imputado_mima = pd.read_csv(ruta_mima, parse_dates=['date'])
    series_para_resumen = [('híbrida (este notebook)', imputado_hibrido), ('MIMA', imputado_mima)]
else:
    imputado_mima = None
    series_para_resumen = [('híbrida (este notebook)', imputado_hibrido)]
    print("La serie imputada con MIMA todavía no existe. Con la base por zona ya guardada (sección 3), córrala así:")
    print("  cd data_processed/mima && python mima_pipeline.py --reuse-model   (usa el modelo entrenado incluido;")
    print("  sin la opción, re-entrena desde cero en unas dos horas de CPU). El notebook 01 la necesita.")
    print("/ The MIMA-imputed series does not exist yet. With the zone base already saved (section 3), run:")
    print("  cd data_processed/mima && python mima_pipeline.py --reuse-model   (uses the included trained model;")
    print("  without the flag it retrains from scratch in about two CPU hours). Notebook 01 needs it.")

print("=" * 70)
print("SERIES HORARIAS IMPUTADAS")
print("=" * 70)
for nombre, df in series_para_resumen:
    cobertura = df['valor'].notna().mean() * 100
    print(f"  {nombre}: {len(df):,} filas | {df['date'].min()} a {df['date'].max()} | "
          f"cobertura de la malla completa: {cobertura:.1f}%")
pct_imputado = imputado_hibrido['imputada'].mean() * 100
print(f"  Horas imputadas por la vía híbrida: {pct_imputado:.1f}% (el resto es observado)")
print(f"\nGuardado en: {output_dir / 'datos_horarios_imputados_hibrido.csv.gz'}")

SERIES HORARIAS IMPUTADAS
  híbrida (este notebook): 789,120 filas | 2023-01-01 00:00:00 a 2025-12-31 23:00:00 | cobertura de la malla completa: 100.0%
  MIMA: 789,120 filas | 2023-01-01 00:00:00 a 2025-12-31 23:00:00 | cobertura de la malla completa: 100.0%
  Horas imputadas por la vía híbrida: 2.1% (el resto es observado)

Guardado en: ../data_processed/datos_horarios_imputados_hibrido.csv.gz


## 7. El calendario escolar OFICIAL (fijo, por fecha, con fuentes)

La variable de periodo lectivo sale del calendario oficial de educación básica de la
SEP, NO del dataset. Las fechas se toman de los acuerdos publicados en el Diario
Oficial de la Federación (DOF), que incluyen el calendario gráfico de cada ciclo:

- **Ciclo 2022-23** (Acuerdo 09/06/22, DOF 3-jun-2022): regreso de vacaciones de
  invierno el 9-ene-2023; semana santa del 3 al 14-abr-2023; fin de clases el
  26-jul-2023. Suspensiones: 6-feb, 17-mar (descarga administrativa), 20-mar, 1-may,
  5-may y 15-may-2023.
- **Ciclo 2023-24** (Acuerdo 02/06/23, DOF 26-jun-2023): inicio 28-ago-2023;
  vacaciones de invierno del 18-dic-2023 al 5-ene-2024 (regreso 8-ene-2024); semana
  santa del 25-mar al 5-abr-2024; fin de clases el 16-jul-2024. Suspensiones: 2-nov y
  20-nov-2023; 5-feb, 18-mar, 1-may y 15-may-2024.
- **Ciclo 2024-25** (Acuerdo 10/06/24, DOF 10-jun-2024): inicio 26-ago-2024;
  vacaciones de invierno del 19-dic-2024 al 8-ene-2025 (regreso 9-ene-2025); semana
  santa del 14 al 25-abr-2025; fin de clases el 16-jul-2025. Suspensiones: 16-sep y
  18-nov-2024; 3-feb, 17-mar, 1-may, 5-may y 15-may-2025.
- **Ciclo 2025-26** (Acuerdo 18/06/25, DOF 9-jun-2025): inicio 1-sep-2025;
  vacaciones de invierno desde el 22-dic-2025 (hasta el fin de la ventana de estudio).
  Suspensiones: 16-sep y 17-nov-2025.

Los días de fin de ciclo a inicio del siguiente (receso de verano) cuentan como
vacaciones. Limitación documentada: los viernes de Consejo Técnico Escolar (uno al
mes, sin asistencia de alumnos) no se marcan; el sesgo resultante es conservador
(días sin alumnos contados como lectivos ATENÚAN el contraste escolar, no lo inflan).

In [7]:
fechas_malla = pd.date_range(fecha_min.normalize(), fecha_max.normalize(), freq='D')
calendario = pd.DataFrame({'date': fechas_malla})

ciclos_lectivos = [('2023-01-09', '2023-07-26'),   # 2022-23 (mitad en ventana)
                   ('2023-08-28', '2024-07-16'),   # 2023-24
                   ('2024-08-26', '2025-07-16'),   # 2024-25
                   ('2025-09-01', '2026-07-15')]   # 2025-26 (hasta el fin de la ventana)
vacaciones = [('2023-04-03', '2023-04-14'),        # semana santa 2023
              ('2023-12-18', '2024-01-05'),        # invierno 2023-24
              ('2024-03-25', '2024-04-05'),        # semana santa 2024
              ('2024-12-19', '2025-01-08'),        # invierno 2024-25
              ('2025-04-14', '2025-04-25'),        # semana santa 2025
              ('2025-12-22', '2026-01-06')]        # invierno 2025-26
suspensiones = ['2023-02-06', '2023-03-17', '2023-03-20', '2023-05-01', '2023-05-05',
                '2023-05-15', '2023-11-02', '2023-11-20', '2024-02-05', '2024-03-18',
                '2024-05-01', '2024-05-15', '2024-09-16', '2024-11-18', '2025-02-03',
                '2025-03-17', '2025-05-01', '2025-05-05', '2025-05-15', '2025-09-16',
                '2025-11-17']

en_ciclo = np.zeros(len(calendario), dtype=bool)
for inicio, fin in ciclos_lectivos:
    en_ciclo |= (calendario['date'] >= inicio) & (calendario['date'] <= fin)
en_vacaciones = np.zeros(len(calendario), dtype=bool)
for inicio, fin in vacaciones:
    en_vacaciones |= (calendario['date'] >= inicio) & (calendario['date'] <= fin)

calendario['lectivo'] = (en_ciclo & ~en_vacaciones).astype(int)
calendario['suspension'] = calendario['date'].isin(pd.to_datetime(suspensiones)).astype(int)
calendario['dow'] = calendario['date'].dt.dayofweek
calendario['dia_clases'] = ((calendario['lectivo'] == 1) & (calendario['dow'] < 5)
                            & (calendario['suspension'] == 0)).astype(int)

def clasificar_tipo_dia(fila):
    """
    Variable categórica: hábil/fin de semana × lectivo/vacaciones.
    """
    habil = fila['dow'] < 5
    lectivo = fila['lectivo'] == 1
    if habil and lectivo:
        return 'habil_lectivo'
    if habil and not lectivo:
        return 'habil_vacaciones'
    if not habil and lectivo:
        return 'finde_lectivo'
    return 'finde_vacaciones'

calendario['tipo_dia'] = calendario.apply(clasificar_tipo_dia, axis=1)

print("=" * 70)
print("CALENDARIO OFICIAL SEP CONSTRUIDO")
print("=" * 70)
print(f"Ventana: {fechas_malla.min().date()} a {fechas_malla.max().date()} ({len(calendario)} días)")
print(f"Días lectivos (bloques): {int(calendario['lectivo'].sum())} | "
      f"días efectivos de clase (hábil, sin suspensión): {int(calendario['dia_clases'].sum())}")
print("\nConteo por tipo de día:")
print(calendario['tipo_dia'].value_counts())

calendario.to_csv(output_dir / 'calendario_oficial_sep.csv', index=False)
print(f"\nGuardado en: {output_dir / 'calendario_oficial_sep.csv'}")

print('\nFINALIZADO')

CALENDARIO OFICIAL SEP CONSTRUIDO
Ventana: 2023-01-01 a 2025-12-31 (1096 días)
Días lectivos (bloques): 884 | días efectivos de clase (hábil, sin suspensión): 607

Conteo por tipo de día:
tipo_dia
habil_lectivo       628
finde_lectivo       256
habil_vacaciones    155
finde_vacaciones     57
Name: count, dtype: int64

Guardado en: ../data_processed/calendario_oficial_sep.csv

FINALIZADO
